# 3. Cached pretrained VLM trajectory embedding

질문: repo의 cached Prismatic VLM graph-node representation에서도 같은 trajectories의 구조가 보이는가?

여기서 보는 것은 optimal expert policy hidden state가 아닙니다. HM3D expert dataset에는 neural hidden state가 없습니다. 이 notebook은 cached VLM/topology graph-node vectors를 2D로 내려 같은 trajectory labels/colors로 읽습니다.

In [ ]:
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")

from pathlib import Path
import sys

repo_root = None
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "pyproject.toml").is_file() and (candidate / "analysis").is_dir():
        repo_root = candidate
        break
if repo_root is None:
    raise RuntimeError("Run this notebook inside the TopoVLM repository.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import importlib
from analysis.code import hm3d_trajectory_notebook
hm3d_trajectory_notebook = importlib.reload(hm3d_trajectory_notebook)

DEFAULT_DATA_ROOT = hm3d_trajectory_notebook.DEFAULT_DATA_ROOT
DEFAULT_RESULT_DIR = hm3d_trajectory_notebook.DEFAULT_RESULT_DIR
load_vlm_node_feature_rows = hm3d_trajectory_notebook.load_vlm_node_feature_rows
pca_2d = hm3d_trajectory_notebook.pca_2d
plot_embedding_trajectories = hm3d_trajectory_notebook.plot_embedding_trajectories
select_scene_trajectory_records = hm3d_trajectory_notebook.select_scene_trajectory_records
trajectory_selection_summary = hm3d_trajectory_notebook.trajectory_selection_summary
save_figure = hm3d_trajectory_notebook.save_figure

DATA_ROOT = repo_root / DEFAULT_DATA_ROOT
records = select_scene_trajectory_records(DATA_ROOT, max_trajectories=6, min_steps=80, min_turns=5)
trajectory_selection_summary(records)


In [ ]:
features, rows = load_vlm_node_feature_rows(records, data_root=DATA_ROOT)
coords, explained = pca_2d(features)
print("vlm_node_features", features.shape, "pca_explained", explained.round(3).tolist())
fig, ax = plot_embedding_trajectories(
    coords,
    rows,
    records,
    title="Cached Prismatic VLM graph-node trajectory PCA",
)
save_figure(fig, "03_vlm_cached_pca.png", result_dir=repo_root / DEFAULT_RESULT_DIR)


읽는 방법: Notebook 1, 2와 같은 `Txx` labels/colors를 사용합니다. Observation PCA와 비교했을 때 trajectory continuity나 split이 더 선명하면, pretrained/cached VLM representation이 branch-relevant structure를 더 잘 보존한다는 candidate evidence가 됩니다. 이 결과만으로 language-conditioned branch sensitivity를 결론내리지는 않습니다. Language effect는 같은 observation에 다른 prompt를 넣는 별도 forward/readout이 필요합니다.